In [2]:
import pandas as pd
import numpy as np
import nfl_data_py as nfl
import matplotlib.pyplot as plt
import seaborn as sns

# Estructura del Notebook
* §1 · *Data Audit*          → shape, dtypes, missings, coverage temporal
* §2 · *Target Variable*     → home win rate por season/week/roof/surface  

# Functions

In [10]:
COLS_TO_DROP = [
    'nfl_detail_id', 'pff', 'ftn', 'away_moneyline', 'home_moneyline',
    'away_spread_odds', 'home_spread_odds','over_odds', 'under_odds'
]

FRANCHISE_MAP = {
    # Rams
    "STL": "LA",
    "LA": "LA",
    "LAR": "LA",

    # Raiders
    "OAK": "LV",
    "LV": "LV",
    "LVR": "LV",

    # Chargers
    "SD": "LAC",
    "LAC": "LAC"
}


# Function to load and preprocess NFL data for given seasons

def load_nfl_data(seasons, cols_to_drop = COLS_TO_DROP, franchise_map = FRANCHISE_MAP):
    
    df = nfl.import_schedules(seasons)
    df = df.drop(columns=cols_to_drop)
    df["home_team"] = df["home_team"].replace(franchise_map)
    df["away_team"] = df["away_team"].replace(franchise_map)
    df['home_team_win'] = (df['home_score'] > df['away_score']).astype(int)
    
    
    return df

In [25]:
# ── Constantes del sistema ─────────────────────────────────────────────────────

ELO_INICIAL    = 1500
K_REGULAR      = 20
K_PLAYOFFS     = 24
REGRESION      = 0.33
MEDIA_GLOBAL   = 1505

GAME_TYPE_K = {
    'REG': K_REGULAR,
    'WC' : K_PLAYOFFS,
    'DIV': K_PLAYOFFS,
    'CON': K_PLAYOFFS,
    'SB' : K_PLAYOFFS,
}


# ── Función 1: HOME_ADVANTAGE ──────────────────────────────────────────────────

def estimate_home_advantage(df, last_n_seasons=None, game_types=['REG']):
    """
    Calibra HOME_ADVANTAGE desde el home win rate observado.
    HA = -400 * log10((1 / P_home) - 1)
    """
    mask = df['game_type'].isin(game_types)

    if last_n_seasons is not None:
        max_season = df['season'].max()
        min_season = max_season - last_n_seasons + 1
        mask = mask & (df['season'] >= min_season)

    subset = df[mask]
    observed_rate = subset['home_team_win'].mean()
    ha = -400 * np.log10((1 / observed_rate) - 1)

    return {
        'home_advantage': round(ha, 1),
        'home_win_rate' : round(observed_rate, 4),
        'n_games'       : len(subset),
        'seasons'       : (subset['season'].min(), subset['season'].max())
    }


# ── Función 2: probabilidad esperada ──────────────────────────────────────────

def expected_win_prob(elo_home, elo_away, home_advantage):
    """
    Calcula la probabilidad de victoria del equipo local.

    P_home = 1 / (1 + 10^(-((R_home - R_away + HA) / 400)))

    Parameters
    ----------
    elo_home       : float — rating ELO del equipo local
    elo_away       : float — rating ELO del equipo visitante
    home_advantage : float — puntos ELO de ventaja local (0 para SB)

    Returns
    -------
    float — probabilidad de victoria del local [0, 1]
    """
    diff = (elo_home - elo_away + home_advantage) / 400
    return 1 / (1 + 10 ** (-diff))


# ── Función 3: actualización de ratings ───────────────────────────────────────

def update_elo(elo_home, elo_away, result, k, home_advantage):
    """
    Actualiza los ratings ELO después de un partido.

    New_ELO_home = ELO_home + K * (S - P_home)
    New_ELO_away = ELO_away + K * ((1-S) - (1-P_home))

    Parameters
    ----------
    elo_home       : float — rating ELO del equipo local pre-partido
    elo_away       : float — rating ELO del equipo visitante pre-partido
    result         : int   — 1 si ganó local, 0 si ganó visitante
    k              : float — K-factor del partido
    home_advantage : float — puntos ELO de ventaja local (0 para SB)

    Returns
    -------
    tuple (new_elo_home, new_elo_away)
    """
    p_home = expected_win_prob(elo_home, elo_away, home_advantage)

    delta = k * (result - p_home)

    return (
        round(elo_home + delta, 4),
        round(elo_away - delta, 4),
    )


# ── Función 4: regresión entre temporadas ─────────────────────────────────────

def apply_season_regression(elo_ratings):
    """
    Jala todos los ratings hacia la media global al inicio de cada temporada.

    ELO_inicio = (1 - α) * ELO_final + α * MEDIA_GLOBAL

    Parameters
    ----------
    elo_ratings : dict — {team: elo_rating}

    Returns
    -------
    dict — {team: elo_rating_regresado}
    """
    return {
        team: round((1 - REGRESION) * elo + REGRESION * MEDIA_GLOBAL, 4)
        for team, elo in elo_ratings.items()
    }


# ── Función 5: sistema completo ───────────────────────────────────────────────

def run_elo_system(df, home_advantage):
    """
    Corre el sistema ELO sobre todo el histórico partido a partido.

    Parameters
    ----------
    df             : pd.DataFrame — schedules limpio (output de data_loader)
    home_advantage : float        — puntos ELO de ventaja local para REG/WC/DIV/CON
                                    SB siempre usa HA=0

    Returns
    -------
    history_df : pd.DataFrame — ELO de cada equipo antes y después de cada partido
    final_elos : dict         — {team: elo_final} al cierre del último season
    """
    # Ordenar por temporada y semana para garantizar orden cronológico
    df = df.sort_values(['season', 'week']).reset_index(drop=True)

    # Inicializar ratings
    teams     = set(df['home_team']).union(set(df['away_team']))
    elo_ratings = {team: ELO_INICIAL for team in teams}

    records = []
    current_season = None

    for _, row in df.iterrows():

        season    = row['season']
        game_type = row['game_type']
        home_team = row['home_team']
        away_team = row['away_team']
        result    = row['home_team_win']

        # ── Regresión al inicio de cada nueva temporada ──────────────────────
        if season != current_season:
            if current_season is not None:
                elo_ratings = apply_season_regression(elo_ratings)
            current_season = season

        # ── Parámetros del partido ────────────────────────────────────────────
        k  = GAME_TYPE_K[game_type]
        ha = 0.0 if game_type == 'SB' else home_advantage

        elo_home_pre = elo_ratings[home_team]
        elo_away_pre = elo_ratings[away_team]

        p_home = expected_win_prob(elo_home_pre, elo_away_pre, ha)

        # ── Actualizar ratings ────────────────────────────────────────────────
        elo_home_post, elo_away_post = update_elo(
            elo_home_pre, elo_away_pre, result, k, ha
        )

        elo_ratings[home_team] = elo_home_post
        elo_ratings[away_team] = elo_away_post

        # ── Registrar ─────────────────────────────────────────────────────────
        records.append({
            'season'        : season,
            'week'          : row['week'],
            'game_type'     : game_type,
            'home_team'     : home_team,
            'away_team'     : away_team,
            'elo_home_pre'  : elo_home_pre,
            'elo_away_pre'  : elo_away_pre,
            'home_advantage': ha,
            'p_home'        : round(p_home, 4),
            'result'        : result,
            'elo_home_post' : elo_home_post,
            'elo_away_post' : elo_away_post,
        })

    history_df = pd.DataFrame(records)
    final_elos = elo_ratings

    return history_df, final_elos

In [41]:
# ── Cargar datos ──────────────────────────────────────────────────────────────
df = load_nfl_data(range(1999, 2026))

# ── Calibrar HOME_ADVANTAGE ───────────────────────────────────────────────────
ha_result = estimate_home_advantage(df)
HOME_ADVANTAGE = ha_result['home_advantage']
print(f"HOME_ADVANTAGE calibrado: {HOME_ADVANTAGE}")

# ── Correr sistema ELO ────────────────────────────────────────────────────────
history_df, final_elos = run_elo_system(df, home_advantage=HOME_ADVANTAGE)

HOME_ADVANTAGE calibrado: 42.2


In [42]:
final_elos_df = (
    pd.Series(final_elos)
      .reset_index()
      .rename(columns={'index': 'team', 0: 'elo_final'})
      .sort_values('elo_final', ascending=False)
)

print("\nTop 10 equipos al cierre de 2025:")
print(final_elos_df.head(10).to_string(index=False))


Top 10 equipos al cierre de 2025:
team  elo_final
 SEA  1641.1072
 BUF  1605.8581
 DEN  1595.3623
 PHI  1593.8584
  LA  1587.5497
 HOU  1577.2630
  NE  1572.3085
 DET  1567.9271
  SF  1563.9628
 MIN  1554.4757


In [44]:
# ¿El ganador del Super Bowl tenía ELO > 1500 antes del partido?
# Es decir, ¿era un equipo "bueno" aunque no fuera el mejor?

sb_games = history_df[history_df['game_type'] == 'SB'].copy()
sb_games['winner_elo_pre'] = sb_games.apply(
    lambda r: r['elo_home_pre'] if r['result'] == 1 else r['elo_away_pre'],
    axis=1
)
sb_games['loser_elo_pre'] = sb_games.apply(
    lambda r: r['elo_away_pre'] if r['result'] == 1 else r['elo_home_pre'],
    axis=1
)

print(sb_games[['season', 'winner_elo_pre', 'loser_elo_pre', 'p_home']].to_string())

      season  winner_elo_pre  loser_elo_pre  p_home
258     1999       1592.0608      1617.5397  0.5366
517     2000       1604.7787      1578.8127  0.4627
776     2001       1565.3581      1641.4506  0.3922
1043    2002       1601.8655      1597.1694  0.5068
1310    2003       1651.6266      1558.0730  0.6315
1577    2004       1687.8347      1635.0043  0.4246
1844    2005       1642.2331      1603.2656  0.5558
2111    2006       1647.7495      1595.2651  0.4250
2378    2007       1574.9065      1709.2710  0.6843
2645    2008       1616.6256      1538.1636  0.3890
2912    2009       1590.5560      1662.9428  0.6027
3179    2010       1592.8792      1616.2074  0.4665
3446    2011       1585.9047      1656.3998  0.6001
3713    2012       1614.2347      1604.8588  0.4865
3980    2013       1621.0437      1625.9258  0.5070
4247    2014       1639.7967      1653.8802  0.5203
4514    2015       1633.0165      1639.7610  0.4903
4781    2016       1672.2856      1570.6077  0.3577
5048    2017